# 📈 Generate Outputs — Kaggle (Step 8/8)
**Generates final figures and tables for the paper.**

## ⚙️ Setup
1. **Datasets** — Add all 12 datasets (Input → Add Data)
2. **Secret** — Add `HF_TOKEN` secret
3. **Accelerator** — GPU T4 x2 (or None, it's mostly plotting)
4. **Run All**

**Sequence**: Tiny → Base → Large → Baselines → Ablation → PaperEvals → LOGO → **Outputs**

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 0: Clone Repo + Install Dependencies
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

REPO_URL  = "https://github.com/MIHMahmudEli/ai-image-detection-research.git"
CLONE_DIR = Path("/kaggle/working/ai-image-detection-research")

if not CLONE_DIR.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--rebase"], check=False)

os.chdir(str(CLONE_DIR))
sys.path.insert(0, str(CLONE_DIR / "model"))

subprocess.run([sys.executable, "-m", "pip", "install",
    "huggingface_hub", "scikit-learn", "matplotlib", "seaborn",
    "-q", "--disable-pip-version-check"], check=False)

print(f"Project root: {CLONE_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Environment & HuggingFace Download
# ═══════════════════════════════════════════════════════════
import os, sys, json, shutil
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path("/kaggle/working/ai-image-detection-research/model")))
from src.kaggle_utils import KaggleEnv
env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

print("Downloading results from HF...")
if env.hf_token:
    try:
        from huggingface_hub import snapshot_download
        local_dir = PROJECT_ROOT / "paper" / "result"
        snapshot_download(repo_id=env.hf_results_repo, repo_type="model", local_dir=local_dir, token=env.hf_token)
        print(f"Downloaded results to {local_dir}")
    except Exception as e:
        print(f"Error downloading results: {e}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Generate Final Paper Views
# ═══════════════════════════════════════════════════════════
from src.visualize import plot_ablation_bar_chart

RES_DIR = PROJECT_ROOT / "paper" / "result"
OUT = PROJECT_ROOT / "paper" / "result" / "final_paper_figures"
OUT.mkdir(parents=True, exist_ok=True)

# Ablation Summary
abl_csv = RES_DIR / "results" / "ablation" / "table_4_summary.csv"
if abl_csv.exists():
    df = pd.read_csv(abl_csv)
    configs = df['label'].tolist()
    accs = df['test_acc'].tolist()
    plot_ablation_bar_chart(configs, accs, OUT / "ablation_chart.png")
    print("Generated ablation chart.")

# Combine Baseline + LOGO into Markdown Report
report = ["# MFFT Final Paper Metrics\n"]

b_csv = RES_DIR / "results" / "baselines" / "baseline_summary.csv"
if b_csv.exists():
    report.append("## Baseline Comparison")
    report.append(pd.read_csv(b_csv).to_markdown())

l_csv = RES_DIR / "results" / "logo" / "logo_results.csv"
if l_csv.exists():
    report.append("\n## LOGO Generalization")
    report.append(pd.read_csv(l_csv).to_markdown())

with open(OUT / "final_report.md", "w") as f:
    f.write("\n".join(report))

env.upload_to_hf(OUT, env.hf_results_repo, "results/final_paper_figures")
print("\n✅ All pipeline stages complete! Results uploaded to HF.")